In [ ]:
!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-text-splitters
!pip install -q chromadb
!pip install -q sentence-transformers
!pip install -q langchain-huggingface
!pip install -q google-generativeai
!pip install -q pypdf

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph 1.2.2 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.2.43 which is incompatible.
langchain-google-genai 4.2.4 requires langchain-core<2.0.0,>=1.3.2, but you have langchain-core 0.2.43 which is incompatible.
langchain-classic 1.0.7 requires langchain-core<2.0.0,>=1.3.3, but you have langchain-core 0.2.43 which is incompatible.
langchain-classic 1.0.7 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.2.4 which is incompatible.
langgraph-prebuilt 1.1.0 requires langchain-core>=1.3.1, but you have langchain-core 0.2.43 which is incompatible.


In [ ]:
import os

from langchain_community.document_loaders import (
    DirectoryLoader,
    PyPDFLoader
)

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter
)

from langchain_huggingface import (
    HuggingFaceEmbeddings
)

from langchain_community.vectorstores import (
    Chroma
)

import google.generativeai as genai

In [ ]:
os.makedirs("/content/pdfs", exist_ok=True)

print("Folder Created!")

Folder Created!


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving RAG_Complete_Guide.pdf to RAG_Complete_Guide.pdf


In [ ]:
from google.colab import files

uploaded = files.upload()


Saving AI_Notes.pdf to AI_Notes.pdf


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving Research_Paper.pdf to Research_Paper.pdf


In [ ]:
import shutil

for file_name in uploaded.keys():
    shutil.move(
        f"/content/{file_name}",
        f"/content/pdfs/{file_name}"
    )

print("All PDFs moved successfully!")

All PDFs moved successfully!


In [ ]:
print(os.listdir("/content/pdfs"))

['Research_Paper.pdf', 'RAG_Complete_Guide.pdf', 'AI_Notes.pdf']


In [ ]:
loader = DirectoryLoader(
    "/content/pdfs",
    glob="*.pdf",
    loader_cls=PyPDFLoader
)

docs = loader.load()

print("Total Documents:", len(docs))

Total Documents: 11


In [ ]:
print(docs[0].metadata)

{'source': '/content/pdfs/Research_Paper.pdf', 'page': 0}


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(docs)

print("Total Chunks:", len(chunks))

Total Chunks: 29


In [ ]:
print(chunks[0].metadata)

print("\n")

print(chunks[0].page_content[:300])

{'source': '/content/pdfs/Research_Paper.pdf', 'page': 0}


RESEARCH PAPER: RETRIEV AL 
AUGMENTED GENERATION (RAG) 
Abstract 
Retrieval-Augmented Generation (RAG) combines information retrieval systems with large 
language models to improve the accuracy and reliability of generated responses. 
Introduction 
Traditional language models generate responses base


In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding Model Loaded!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding Model Loaded!


In [ ]:
vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./multi_pdf_db"
)

print("Vector Database Created!")

Vector Database Created!


In [ ]:
retriever = vectordb.as_retriever(
    search_kwargs={"k": 4}
)

print("Retriever Ready!")

Retriever Ready!


In [ ]:
genai.configure(
    api_key="Your API_Key"
)

model = genai.GenerativeModel(
    "gemini-2.5-flash"
)

print("Gemini Connected!")

Gemini Connected!


In [ ]:
def ask_rag(question):

    retrieved_docs = retriever.invoke(question)

    context = "\n\n".join(
        [doc.page_content for doc in retrieved_docs]
    )

    prompt = f"""
You are a helpful AI assistant.

Use ONLY the context below.

If the answer is not found in the context,
say:

I don't know based on the provided documents.

Context:
{context}

Question:
{question}

Answer:
"""

    response = model.generate_content(prompt)

    sources = []

    for doc in retrieved_docs:

        source = doc.metadata.get(
            "source",
            "Unknown"
        )

        page = doc.metadata.get(
            "page",
            0
        ) + 1

        sources.append(
            f"{source} (Page {page})"
        )

    sources = list(set(sources))

    return {
        "answer": response.text,
        "sources": sources
    }

In [ ]:
result = ask_rag(
    "What is Machine Learning?"
)

print("Answer:\n")
print(result["answer"])

print("\nSources:\n")

for source in result["sources"]:
    print(source)

Answer:

Machine Learning is a subset of AI that enables computers to learn patterns from data without being explicitly programmed.

Sources:

/content/pdfs/AI_Notes.pdf (Page 1)


In [ ]:
result = ask_rag(
    "What are the advantages of RAG?"
)

print("Answer:\n")
print(result["answer"])

print("\nSources:\n")

for source in result["sources"]:
    print(source)

Answer:

The advantages of RAG (Retrieval-Augmented Generation) are:
*   It gives your AI a brain filled with your data.
*   It searches your private documents first, then generates an accurate answer based on facts.
*   It results in no hallucinations.
*   It has no API costs.
*   It is 100% offline/local.

Sources:

/content/pdfs/RAG_Complete_Guide.pdf (Page 1)
/content/pdfs/RAG_Complete_Guide.pdf (Page 6)


In [ ]:
result = ask_rag(
    "What is ChromaDB?"
)

print("Answer:\n")
print(result["answer"])

print("\nSources:\n")

for source in result["sources"]:
    print(source)

Answer:

ChromaDB is a Vector Database that stores and searches text embeddings fast.

Sources:

/content/pdfs/RAG_Complete_Guide.pdf (Page 1)
/content/pdfs/RAG_Complete_Guide.pdf (Page 4)
